# 🔬 Multi-Agent Research System: Planner, Parallel Search and a Quality Loop

This capstone notebook combines the multi-agent patterns of the earlier modules into a single, working research
pipeline: a **planner** splits the topic into search queries, multiple **search agents** run **in parallel** via
the `Send` API and append to one shared **findings list**, an **analyst** and **report writer** synthesize those
findings, and a **quality checker** drives an **iterative refinement loop** until the report is good enough to ship.

## Learning Objectives
In this notebook, you will learn:
1. **A one-shot planner** - how a planning node decomposes a broad topic into targeted, parallelizable sub-tasks
   (it plans once; unlike a supervisor it never sees results or picks who acts next)
2. **Parallel Execution with `Send`** - how to dynamically fan out work to multiple agent instances and fan the results back in
3. **A shared findings list** - how independent agents accumulate results into one key (`findings`) through a reducer
4. **Iterative Refinement Loops** - how a quality-gate conditional edge can send work back for revision until it meets a bar, with a safety cap on iterations
5. **End-to-End Graph Composition** - how to wire all of the above nodes and edges into one compiled `StateGraph`

## Where this fits
`13_multi_agent_research_summarization.ipynb` routed each question to exactly one specialist. Here the number
of workers is decided at **run time** by the planner, they run at the same time, and the graph has a
real cycle. The parallel mechanics themselves (supersteps, joins, reducers, `Send`) are taught with zero-LLM stub
graphs in [`01_parallel_agents.ipynb`](../../m26_collaboration_parallel_agents/notebooks/05a_parallel_agents.ipynb) (module 09); this notebook applies them. This is session 5 of the
[Supervisor to Swarm study route](../../../../docs/source/08/supervisor-to-swarm/README.md).

## Prerequisites
- Parallel execution with `Send` (`modules/09_async_and_parallel_multi_agent/notebooks/01_parallel_agents.ipynb`), as
  reading; for contrast, the supervisor pattern (`modules/06_supervisor_and_hierarchical_agents/notebooks/01_supervisor_pattern.ipynb`:
  a supervisor decides who acts next after every result) and a real blackboard, whose agents read each other's
  entries (`modules/08_collaboration_and_control_patterns/notebooks/01_blackboard_intro.ipynb`)
- Comfort with LangGraph `StateGraph`, conditional edges and structured output (Pydantic)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`
- Internet access for the DuckDuckGo search (`ddgs`, no API key needed)

---
## 📦 Step 1: Environment Setup

The setup cell loads `OPENAI_API_KEY` and imports the building blocks.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

import json
import operator
from datetime import date
from typing import Literal

from ddgs import DDGS
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.types import Send
from pydantic import BaseModel, Field
from typing_extensions import Annotated, TypedDict
from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

---
## 🤖 Part 1: Two Models

Two models from the same factory: a deterministic `llm` (temperature `0`) for planning, extraction, analysis and
scoring, and a `creative_llm` (temperature `0.7`) for drafting the report text. The non-zero temperature also
matters for the loop: a revision at temperature 0 tends to repeat the previous draft.

In [ ]:
# ============================================================================
# LLM INITIALIZATION: a deterministic model and a creative one
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini", temperature=0)            # planning, extraction, analysis, scoring
creative_llm = get_openai_llm(model="gpt-4o-mini", temperature=0.7)  # drafting the report

print(f"✅ LLM initialized: {llm.model_name} (temperature={llm.temperature})")
print(f"✅ Creative LLM initialized: {creative_llm.model_name} (temperature={creative_llm.temperature})")

---
## 🎛️ Part 2: Settings

The numbers that decide the size and cost of a run, in one place. The code enforces each of them: the plan is
cut to `NUM_QUERIES`, each search keeps `RESULTS_PER_QUERY` hits, and the quality gate stops after `MAX_DRAFTS`
drafts whatever the reviewer says.

In [ ]:
# ============================================================================
# SETTINGS
# ============================================================================
NUM_QUERIES = 3        # search queries the planner writes = parallel search agents
RESULTS_PER_QUERY = 3  # DuckDuckGo hits each search agent reads
QUALITY_THRESHOLD = 0.7  # every review criterion must reach this for approval
MAX_DRAFTS = 2         # hard cap on report drafts (1 draft + 1 revision)
REQUIRED_SECTIONS = ["Executive Summary", "Key Findings", "Analysis", "Recommendations"]

print(f"Up to {NUM_QUERIES} parallel searches x {RESULTS_PER_QUERY} results, at most {MAX_DRAFTS} drafts")

---
## 🧠 Part 3: State Schemas

Two `TypedDict` schemas drive this graph. `ResearchState` is the main graph state: it carries the shared
**findings list** (`findings`) that every search agent writes into. `SearchTaskState` is the smaller, per-task
state that each parallel `search_agent` instance receives from the `Send` API: a worker only needs its own
query, not the whole research state.

### Key Concepts:
- **`Annotated[..., operator.add]`**: tells LangGraph to *append* new findings to the existing list rather than
  overwrite it. This is what lets parallel branches write to one list (Part 3 of
  `01_parallel_agents.ipynb`, module 09, shows what happens without it)
- **`Annotated[..., add_messages]`**: the standard LangGraph reducer for accumulating an activity trace
- **`iteration`**: counts completed reviews; the quality gate compares it with `MAX_DRAFTS`

In [ ]:
# ============================================================================
# STATE SCHEMAS: ResearchState and SearchTaskState
# ============================================================================
class ResearchState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    topic: str
    search_queries: list[str]
    findings: Annotated[list[dict], operator.add]  # the shared findings list: parallel writers append
    analysis: str
    report: str
    quality_score: float
    quality_feedback: str
    missing_sections: list[str]
    iteration: int


# State for individual search tasks (used with Send API)
class SearchTaskState(TypedDict):
    search_query: str
    findings: Annotated[list[dict], operator.add]


def initial_state(topic: str) -> dict:
    return {
        "messages": [], "topic": topic, "search_queries": [], "findings": [], "analysis": "", "report": "",
        "quality_score": 0.0, "quality_feedback": "", "missing_sections": [], "iteration": 0,
    }


print("✅ ResearchState:", ", ".join(ResearchState.__annotations__))
print("✅ SearchTaskState:", ", ".join(SearchTaskState.__annotations__))

---
## 🤖 Part 4: Agent Nodes

Each node below is a single step of the research pipeline: planning, searching, synthesizing, and writing.

### 4.1 🗂️ `planner`

Plans the research, once: structured output returns a `ResearchPlan` with a list of search queries covering different
angles of the topic. The model is asked for `NUM_QUERIES` of them and the code keeps at most that many, so the
fan-out size never depends on the model counting correctly. Today's date is in the prompt so that queries about
"recent" developments target the right period. It runs once, before any search, and never sees a result: a
planner, not a supervisor.

In [ ]:
# ============================================================================
# PLANNER: Plans Research by Generating Search Queries
# ============================================================================
class ResearchPlan(BaseModel):
    queries: list[str] = Field(description=f"Exactly {NUM_QUERIES} web search queries, each covering a different angle")


def planner(state: ResearchState) -> dict:
    """Plans research by generating targeted search queries."""
    plan = llm.with_structured_output(ResearchPlan).invoke([
        SystemMessage(content=(
            f"You are a research planner. Today is {date.today():%d %B %Y}. Given a topic, write exactly "
            f"{NUM_QUERIES} specific web search queries that cover different angles of it."
        )),
        HumanMessage(content=f"Research topic: {state['topic']}"),
    ])
    queries = plan.queries[:NUM_QUERIES]  # the cap is enforced here, not trusted to the model
    return {
        "search_queries": queries,
        "messages": [AIMessage(content=f"[PLANNER]: Planned {len(queries)} research queries: {queries}", name="planner")],
    }


print("✅ planner defined")

### 4.2 🔎 `search_agent`

Executes a single search query: a real DuckDuckGo search, then the model extracts 2-3 findings, each tied to
the URL of the result it came from. Code then tags every finding with its `source_query` and checks that its
URL is one the search actually returned (`url_verified`), so a made-up source is visible. This node is designed
to be launched **in parallel**, one instance per query, via the `Send` API.

In [ ]:
# ============================================================================
# SEARCH AGENT: Executes One Search Query (Runs in Parallel via Send API)
# ============================================================================
class Finding(BaseModel):
    title: str = Field(description="Short title of the finding")
    detail: str = Field(description="One or two sentences with the specific fact")
    source_url: str = Field(description="URL of the search result this finding comes from")


class SearchFindings(BaseModel):
    findings: list[Finding] = Field(description="2-3 key findings, each supported by one search result")


def web_search(query: str) -> list[dict]:
    """DuckDuckGo text search; an empty list if the search fails."""
    try:
        return DDGS().text(query, max_results=RESULTS_PER_QUERY)
    except Exception as e:  # network errors, rate limits
        print(f"⚠️ search failed for {query!r}: {e}")
        return []


def search_agent(state: SearchTaskState) -> dict:
    """
    Executes one search query and returns findings.
    Each instance runs in parallel via the Send API.
    """
    query = state["search_query"]
    results = web_search(query)
    if not results:
        return {"findings": [{"title": query, "detail": "No search results.", "source_url": "",
                              "source_query": query, "url_verified": False}]}

    formatted = "\n\n".join(f"URL: {r['href']}\nTitle: {r['title']}\n{r['body']}" for r in results)
    extracted = llm.with_structured_output(SearchFindings).invoke([
        SystemMessage(content="You are a web research agent. Extract 2-3 key findings from these search results only."),
        HumanMessage(content=f"Search query: {query}\n\nSearch results:\n{formatted}"),
    ])
    urls = {r["href"] for r in results}
    findings = [
        {**f.model_dump(), "source_query": query, "url_verified": f.source_url in urls}  # tag + check in code
        for f in extracted.findings
    ]
    return {"findings": findings}


print("✅ search_agent defined")

### 4.3 🚀 `dispatch_searches`

The fan-out edge: dynamically creates one `Send("search_agent", ...)` per planned query, so all searches run
concurrently instead of sequentially. The number of `Send`s is the length of a list the planner produced:
it is unknown when the graph is built.

In [ ]:
# ============================================================================
# DISPATCH SEARCHES: Dynamic Fan-Out via the Send API
# ============================================================================
def dispatch_searches(state: ResearchState) -> list[Send]:
    """Dynamically create parallel search tasks using Send API."""
    return [Send("search_agent", {"search_query": query, "findings": []}) for query in state["search_queries"]]


print("✅ dispatch_searches defined")

### 4.4 🧩 `analyst`

Reads every finding accumulated in the shared findings list (`state["findings"]`) and synthesizes them into
themes, gaps/contradictions, and key insights. It runs once, after all search agents have finished.

In [ ]:
# ============================================================================
# ANALYST: Synthesizes the Shared Findings List
# ============================================================================
def analyst(state: ResearchState) -> dict:
    """Reads all findings from the shared list and synthesizes."""
    findings_text = json.dumps(state["findings"], indent=2)
    response = llm.invoke([
        SystemMessage(content=(
            "You are a research analyst. Synthesize the collected findings into a clear analysis. Identify:\n"
            "1. Key themes across all findings\n2. Any contradictions or gaps\n3. The most important insights\n\n"
            "Write 2-3 paragraphs, using only the findings."
        )),
        HumanMessage(content=f"Research topic: {state['topic']}\n\nCollected findings:\n{findings_text}"),
    ])
    return {
        "analysis": response.content,
        "messages": [AIMessage(content=f"[ANALYST]: {response.content}", name="analyst")],
    }


print("✅ analyst defined")

### 4.5 ✍️ `report_writer`

Drafts a structured report with four fixed `##` sections using the higher-temperature `creative_llm`. On a
revision it receives the **previous draft and the reviewer's feedback**, so it improves the draft instead of
starting over. It gets all findings, not just the first few.

In [ ]:
# ============================================================================
# REPORT WRITER: Drafts (or Revises) the Structured Research Report
# ============================================================================
def report_writer(state: ResearchState) -> dict:
    """Writes a structured research report from the analysis."""
    revision = state["iteration"] > 0
    revision_note = ""
    if revision:
        revision_note = (
            f"\n\nThis is revision #{state['iteration']}. Improve the previous draft below and address this "
            f"feedback: {state['quality_feedback']}\n\nPrevious draft:\n{state['report']}"
        )

    sections = "\n".join(f"## {s}" for s in REQUIRED_SECTIONS)
    response = creative_llm.invoke([
        SystemMessage(content=(
            "You are a report writer. Produce a well-structured markdown research report with exactly these "
            f"section headings:\n{sections}\n\n"
            "Executive Summary: 2-3 sentences. Key Findings: bullet points. Analysis: 1-2 paragraphs. "
            "Recommendations: 3 specific, actionable items. Be specific. Reply with the report only."
        )),
        HumanMessage(content=(
            f"Topic: {state['topic']}\n\nAnalysis:\n{state['analysis']}\n\n"
            f"Raw findings:\n{json.dumps(state['findings'], indent=2)}{revision_note}"
        )),
    ])
    return {
        "report": response.content,
        "messages": [AIMessage(content=f"[REPORT WRITER]: Report {'revised' if revision else 'drafted'}.", name="report_writer")],
    }


print("✅ report_writer defined")

---
## 🔍 Part 5: Quality Review Loop

The report isn't automatically final. A `quality_checker` node reviews it and writes scores to the state; a
`quality_gate` conditional edge (a pure function, no model call) decides whether to approve it or send it back
to `report_writer` for another pass.

### Key Insight:
> Iterative refinement loops need a forced exit condition that **code** enforces. The reviewer is never asked to
> "be lenient" on later passes; instead `quality_gate` ends the loop once `iteration` reaches `MAX_DRAFTS`,
> whatever the scores say. Part 6 proves this with a reviewer that never approves.

### 5.1 📊 `QualityReview` and `quality_checker`

`QualityReview` is the Pydantic schema used with `.with_structured_output()`. Instead of one overall score and
a yes/no from the model, it has **one score per criterion**; the code takes the lowest as `quality_score`. The
one criterion code can check exactly, whether the four required sections are present, is checked in code.

In [ ]:
# ============================================================================
# QUALITY REVIEW SCHEMA: Structured Output for the Reviewer LLM
# ============================================================================
class QualityReview(BaseModel):
    completeness: float = Field(description="0.0-1.0: does it cover the topic and use the findings well?")
    clarity: float = Field(description="0.0-1.0: is it well-written and easy to understand?")
    actionability: float = Field(description="0.0-1.0: are the recommendations specific and actionable?")
    feedback: str = Field(description="The most important specific improvements to make")


print("✅ QualityReview fields:", ", ".join(QualityReview.model_fields))

In [ ]:
# ============================================================================
# QUALITY CHECKER: Scores the Report (the gate decides)
# ============================================================================
def quality_checker(state: ResearchState) -> dict:
    """Reviews the report and records its scores; quality_gate makes the decision."""
    review = llm.with_structured_output(QualityReview).invoke([
        SystemMessage(content=(
            "You are a strict quality reviewer of research reports. Score each criterion from 0.0 to 1.0 "
            "and give specific feedback for improvement."
        )),
        HumanMessage(content=f"Topic: {state['topic']}\n\nReport:\n{state['report']}"),
    ])
    score = min(review.completeness, review.clarity, review.actionability)  # the weakest criterion decides
    missing = [s for s in REQUIRED_SECTIONS if f"## {s}".lower() not in state["report"].lower()]  # checked in code
    feedback = review.feedback + (f" Missing sections: {', '.join(missing)}." if missing else "")
    return {
        "quality_score": score,
        "quality_feedback": feedback,
        "missing_sections": missing,
        "iteration": state["iteration"] + 1,
        "messages": [AIMessage(
            content=(f"[QUALITY CHECK]: completeness {review.completeness:.2f}, clarity {review.clarity:.2f}, "
                     f"actionability {review.actionability:.2f}, missing sections {missing or 'none'}: {feedback}"),
            name="quality_checker",
        )],
    }


print("✅ quality_checker defined")

### 5.2 🚦 `quality_gate`

The conditional-edge routing function: approves when the weakest criterion reaches `QUALITY_THRESHOLD` and no
section is missing; otherwise sends the graph back to `report_writer`, unless `MAX_DRAFTS` drafts have already
been reviewed.

In [ ]:
# ============================================================================
# QUALITY GATE: Routes Back to the Writer or Ends the Graph
# ============================================================================
def quality_gate(state: ResearchState) -> Literal["report_writer", "end"]:
    """Route back to writer if quality is insufficient (pure: reads the state only)."""
    approved = state["quality_score"] >= QUALITY_THRESHOLD and not state["missing_sections"]
    if approved or state["iteration"] >= MAX_DRAFTS:  # the cap is enforced here, in code
        return "end"
    return "report_writer"


print("✅ quality_gate defined")

---
## 🏗️ Part 6: Assembling the Graph

`create_research_system` wires every node above into one `StateGraph`: `planner` fans out to parallel
`search_agent`s, which fan back in to `analyst` → `report_writer` → `quality_checker`, with a conditional loop
back to `report_writer` until the quality gate is satisfied.

The nodes are a parameter: `nodes={"search_agent": stub, ...}` replaces any of them. That lets us test the
**wiring** (fan-out size, reducer, loop cap) with stubs, without a single model call.

In [ ]:
# ============================================================================
# CREATE RESEARCH SYSTEM: Build and Compile the Multi-Agent Graph
# ============================================================================
def create_research_system(nodes: dict | None = None):
    """
    Builds the multi-agent research system graph.

    Flow:
    1. Planner plans search queries
    2. Search agents run in parallel (Send API)
    3. Analyst synthesizes findings
    4. Report writer produces report
    5. Quality checker reviews; quality_gate loops back if needed
    """
    impl = {"planner": planner, "search_agent": search_agent, "analyst": analyst,
            "report_writer": report_writer, "quality_checker": quality_checker, **(nodes or {})}
    graph = StateGraph(ResearchState)
    for name, fn in impl.items():
        graph.add_node(name, fn)

    graph.add_edge(START, "planner")
    graph.add_conditional_edges("planner", dispatch_searches, ["search_agent"])  # dynamic fan-out
    graph.add_edge("search_agent", "analyst")  # fan-in: analyst runs once all searches are done
    graph.add_edge("analyst", "report_writer")
    graph.add_edge("report_writer", "quality_checker")
    graph.add_conditional_edges("quality_checker", quality_gate, {"report_writer": "report_writer", "end": END})
    return graph.compile()


system = create_research_system()
print("✅ Graph compiled. Edges:")
for edge in system.get_graph().edges:
    print(f"  {edge.source} -> {edge.target}{'  (conditional)' if edge.conditional else ''}")

### 6.1 🧪 Proving the wiring with stubs (no model calls)

The stubs plan **four** queries (not three) to show the fan-out follows the plan, return two findings per
query, and review every draft with a fixed score. Two runs cover both outcomes of the gate:
- a reviewer that **never approves**: the loop must stop at `MAX_DRAFTS` drafts
- a reviewer that **approves at once**: exactly one draft

In [ ]:
# ============================================================================
# STUB RUNS: FAN-OUT SIZE, REDUCER AND LOOP CAP
# ============================================================================
calls = {"search_agent": 0, "report_writer": 0}
STUB_QUERIES = ["q1", "q2", "q3", "q4"]


def stub_planner(state):
    return {"search_queries": STUB_QUERIES}


def stub_search(state: SearchTaskState):
    calls["search_agent"] += 1
    return {"findings": [{"title": f"{state['search_query']}-{i}", "source_query": state["search_query"]} for i in (1, 2)]}


def stub_analyst(state):
    return {"analysis": "stub analysis"}


def stub_writer(state):
    calls["report_writer"] += 1
    return {"report": "\n".join(f"## {s}" for s in REQUIRED_SECTIONS)}


def stub_reviewer(score):
    return lambda state: {"quality_score": score, "quality_feedback": "stub", "missing_sections": [],
                          "iteration": state["iteration"] + 1}


stubs = {"planner": stub_planner, "search_agent": stub_search, "analyst": stub_analyst, "report_writer": stub_writer}
for label, score, expected_drafts in [("never approves", 0.1, MAX_DRAFTS), ("approves at once", 0.9, 1)]:
    calls.update(search_agent=0, report_writer=0)
    stub_system = create_research_system({**stubs, "quality_checker": stub_reviewer(score)})
    result = stub_system.invoke(initial_state("stub topic"), {"recursion_limit": 25})
    print(f"Reviewer {label} (score {score}):")
    print(f"  {'✅' if calls['search_agent'] == len(STUB_QUERIES) else '❌'} search agents run: {calls['search_agent']} (planned queries: {len(STUB_QUERIES)})")
    print(f"  {'✅' if len(result['findings']) == 2 * len(STUB_QUERIES) else '❌'} findings in the list: {len(result['findings'])} (2 per query, none overwritten)")
    print(f"  {'✅' if {f['source_query'] for f in result['findings']} == set(STUB_QUERIES) else '❌'} every query contributed findings")
    print(f"  {'✅' if calls['report_writer'] == expected_drafts == result['iteration'] else '❌'} drafts written: {calls['report_writer']}, reviews: {result['iteration']} (expected {expected_drafts})")

**Reading the output:** the number of search agents equals the number of planned queries (four here, three in
the real run), and all eight findings survive because the parallel writes go through the `operator.add`
reducer. With a reviewer that never approves, the loop still ends after `MAX_DRAFTS` drafts: that is the code
cap in `quality_gate`, not the model, stopping it.

Why the reducer matters (two parallel writes to a key without one raise `InvalidUpdateError`), and how `Send`
fans out and joins, are shown with stub graphs in Parts 3 and 4 of [`01_parallel_agents.ipynb`](../../m26_collaboration_parallel_agents/notebooks/05a_parallel_agents.ipynb)
(module 09).

---
## 🎬 Part 7: Demos

Two demos exercise the real system: a single search agent in isolation, then the full pipeline with
step-by-step streaming. (`system.invoke(initial_state(topic))` returns the same final state without the live
progress.)

### 7.1 🧪 `demo_individual_search`

Calls `search_agent` directly (outside the graph) to sanity-check a single search step in isolation.

In [ ]:
# ============================================================================
# DEMO: Individual Search Agent (Isolated Test)
# ============================================================================
def demo_individual_search(query: str = "LangGraph multi-agent patterns") -> list[dict]:
    """Demo just the search agent for testing."""
    print(f"Individual Search Agent Test: {query}\n")
    findings = search_agent({"search_query": query, "findings": []})["findings"]
    for f in findings:
        print(f"  {'✅' if f['url_verified'] else '❌'} {f['title']}: {f['detail'][:90]}")
        print(f"     {f['source_url'] or '(no source)'}")
    return findings


individual_findings = demo_individual_search()

**Reading the output:** ✅ marks a finding whose URL is one of the search results the agent actually received;
❌ marks a URL the model produced itself (or a failed search).

### 7.2 📡 `demo_research_with_streaming`

Runs the full graph with `stream_mode=["updates", "values"]`: `updates` prints each node's output as it
completes (each parallel `search_agent` reports separately), `values` gives the final state. The function counts
the node runs so the next cell can check them.

In [ ]:
# ============================================================================
# DEMO: Full Research System with Step-by-Step Streaming
# ============================================================================
def demo_research_with_streaming(topic: str) -> tuple[dict, dict]:
    """Run the research system with step-by-step streaming output; return (final state, node run counts)."""
    print(f"Streaming Research: {topic}\n")
    runs, final = {}, None
    for mode, chunk in system.stream(initial_state(topic), {"recursion_limit": 25}, stream_mode=["updates", "values"]):
        if mode == "values":
            final = chunk
            continue
        for node_name, update in chunk.items():
            runs[node_name] = runs.get(node_name, 0) + 1
            print(f"[{node_name}] completed")
            if update.get("search_queries"):
                print(f"  Planned queries: {update['search_queries']}")
            if update.get("findings"):
                print(f"  Found {len(update['findings'])} findings for: {update['findings'][0]['source_query']}")
            if "quality_score" in update:
                print(f"  Quality score (weakest criterion): {update['quality_score']:.2f}, "
                      f"missing sections: {update['missing_sections'] or 'none'}")
            if update.get("report"):
                print(f"  Report length: {len(update['report'])} chars")
    return final, runs


topic = f"The impact of AI agents on software development in {date.today().year}"
result, node_runs = demo_research_with_streaming(topic)

### 7.3 ✅ Checking the run in code

The claims of this notebook, checked against the streamed run: one search agent per planned query, every query
represented in the findings list, the report written as many times as it was reviewed, and never more than
`MAX_DRAFTS` times.

In [ ]:
# ============================================================================
# CHECK THE RUN, THEN PRINT THE REPORT
# ============================================================================
planned = result["search_queries"]
queries_found = {f["source_query"] for f in result["findings"]}
verified = sum(f["url_verified"] for f in result["findings"])

print(f"{'✅' if len(planned) <= NUM_QUERIES else '❌'} planned queries: {len(planned)} (cap {NUM_QUERIES})")
print(f"{'✅' if node_runs.get('search_agent') == len(planned) else '❌'} search agents run in parallel: {node_runs.get('search_agent')}")
print(f"{'✅' if queries_found == set(planned) else '❌'} findings in the list: {len(result['findings'])} from {len(queries_found)} queries")
print(f"   findings whose URL came from the search results: {verified}/{len(result['findings'])}")
print(f"{'✅' if node_runs.get('report_writer') == result['iteration'] <= MAX_DRAFTS else '❌'} drafts: {node_runs.get('report_writer')}, reviews: {result['iteration']} (cap {MAX_DRAFTS})")
approved = result["quality_score"] >= QUALITY_THRESHOLD and not result["missing_sections"]
print(f"   final verdict: {'approved' if approved else 'stopped by the cap'} (weakest criterion {result['quality_score']:.2f})")

print("\nAgent Activity Log:")
for msg in result["messages"]:
    print(f"  {msg.content[:140]}")

print("\n" + "=" * 60 + "\nFINAL RESEARCH REPORT\n" + "=" * 60)
print(result["report"])

**Reading the output:** the ✅ lines are computed from the state and the streamed updates, not from the model's
wording. The number of drafts depends on the reviewer's scores: `1` means the first draft passed, `2` means one
revision happened (look for `revised` in the activity log). To watch the loop run, raise `QUALITY_THRESHOLD`,
re-run the cells from Part 5 down, and confirm the run still stops at `MAX_DRAFTS`.

---
## 📝 Summary

In this notebook, we combined the multi-agent patterns of the earlier modules into one working research pipeline.

### 1. Patterns Applied
- **A one-shot planner**: `planner` decomposes a broad topic into `NUM_QUERIES` targeted search queries
  (structured output, capped in code); it plans once and never sees the results, which is what separates it from
  a supervisor
- **Parallel execution (`Send` API)**: `dispatch_searches` fans out one `search_agent` per query, running concurrently
- **A shared findings list**: `findings` uses `Annotated[list[dict], operator.add]` so parallel agents safely
  accumulate into one list (why a reducer is needed: Part 3 of
  `09_async_and_parallel_multi_agent/notebooks/01_parallel_agents.ipynb`)
- **Iterative refinement loop**: `quality_checker` scores, `quality_gate` routes the report back to
  `report_writer` (with the previous draft and the feedback) until it passes or hits `MAX_DRAFTS`

### 2. Components Defined
- State: `ResearchState`, `SearchTaskState`
- Nodes: `planner`, `search_agent`, `analyst`, `report_writer`, `quality_checker`
- Routing: `dispatch_searches` (fan-out), `quality_gate` (revise-or-end)
- Schemas: `ResearchPlan`, `Finding`/`SearchFindings`, `QualityReview` (structured output)
- Graph: `create_research_system(nodes=...)`, with swappable nodes for stub tests
- Demos: `demo_individual_search()`, `demo_research_with_streaming()`

### Next Steps
- **`modules/09_async_and_parallel_multi_agent/notebooks/02_async_multi_agent_streaming.ipynb`**: the same
  patterns, async (the next step on the Supervisor to Swarm route)
- **`modules/11_deep_agents_and_harness_engineering/`**: deep agents and harness engineering, the next module
- Taking systems like this to production (observability, evaluation, deployment) is
  [repository 13 (production and observability)](https://github.com/Sourav692/ai-engineering-demystified-13-production-observability)
- As a stretch exercise, add a second parallel branch (e.g., a fact-checking agent) alongside `search_agent` to
  see how the shared findings list scales to more concurrent contributors